# Titanic生存予測：F1スコア最大化（高性能・リーク防止）

## 目的と成功条件

配布された `train_titanic.csv`（712件）だけでモデルの作成・検証・選択を行い、
`test_titanic.csv`（179件）の `Survived` を予測します。正式評価指標は生存者を正例としたF1スコアです。

成功条件は次のとおりです。

- 外部データ、既知のテストラベル、兄弟ディレクトリの予測を使用しない。
- 補完、カテゴリ処理、グループ統計、モデル・閾値・アンサンブル選択を学習fold内だけで行う。
- nested CVで選択バイアスを抑え、単一モデル・weighted blend・cross-fit stackingを比較する。
- `PassengerId,Survived` の2列を持つ179行の提出CSVを3種類生成し、形式を機械的に検査する。

> テスト正解は未知なので、最終F1を事前に保証することはできません。本ノートブックはtrain-onlyの
> nested CVに基づき、未知データに対する期待F1を高める設計です。


## 実験計画

1. 性別・客室等級などの単純な基準を確認する。
2. 行単位特徴に加え、同じ家族・Ticketの学習済み情報をleave-one-outで特徴量化する。
3. CatBoost、LightGBM、XGBoost、Extra Trees、Random Forestをinner CVでOptuna探索する。
4. outer CVでは、inner CVで決めたモデル・閾値・重みだけを使って未使用foldを一度だけ評価する。
5. 全trainで再探索・再学習し、3方式のtest予測を保存する。

`TITANIC_FAST_MODE=1` を指定すると、構造確認用の短縮設定で実行できます。通常実行は性能重視設定です。


In [ ]:
from __future__ import annotations

import os
import random
import re
import time
import warnings
from dataclasses import dataclass
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found.*")

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from catboost import CatBoostClassifier
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")
warnings.filterwarnings("ignore", message="X does not have valid feature names.*")
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
FAST_MODE = os.getenv("TITANIC_FAST_MODE", "0") == "1"
CPU_COUNT = os.cpu_count() or 1
MODEL_THREADS = 1
OPTUNA_JOBS = max(
    1,
    int(os.getenv("TITANIC_OPTUNA_JOBS", str(1 if FAST_MODE else min(4, CPU_COUNT)))),
)


@dataclass(frozen=True)
class ExperimentConfig:
    outer_splits: int
    outer_repeats: int
    inner_splits: int
    nested_trials_boost: int
    nested_trials_forest: int
    final_trials_boost: int
    final_trials_forest: int
    blend_trials: int
    final_cv_seeds: tuple[int, ...]
    final_fit_seeds: tuple[int, ...]


CONFIG = ExperimentConfig(
    outer_splits=3 if FAST_MODE else 5,
    outer_repeats=1 if FAST_MODE else 3,
    inner_splits=3 if FAST_MODE else 4,
    nested_trials_boost=2 if FAST_MODE else 20,
    nested_trials_forest=2 if FAST_MODE else 12,
    final_trials_boost=3 if FAST_MODE else 150,
    final_trials_forest=3 if FAST_MODE else 80,
    blend_trials=20 if FAST_MODE else 500,
    final_cv_seeds=(42,) if FAST_MODE else (42, 123, 2026),
    final_fit_seeds=(42,) if FAST_MODE else (42, 123, 2026, 3407, 7777),
)

print(f"FAST_MODE={FAST_MODE}")
print(f"OPTUNA_JOBS={OPTUNA_JOBS}, MODEL_THREADS={MODEL_THREADS}")
print(CONFIG)


## 1. データ読み込みと契約検査

Notebookを `codex_code` ディレクトリで実行することを前提とします。読み込むCSVはこの2ファイルだけです。
`PassengerId` は提出時の対応付けにのみ使用し、モデル特徴量には含めません。


In [ ]:
DATA_DIR = Path.cwd()
TRAIN_PATH = DATA_DIR / "train_titanic.csv"
TEST_PATH = DATA_DIR / "test_titanic.csv"

if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        "codex_code を作業ディレクトリにして実行してください: "
        f"train={TRAIN_PATH.exists()}, test={TEST_PATH.exists()}"
    )

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

expected_train_columns = [
    "PassengerId", "Survived", "Pclass", "Name", "Sex", "Age",
    "SibSp", "Parch", "Ticket", "Fare", "Cabin", "Embarked",
]
expected_test_columns = [c for c in expected_train_columns if c != "Survived"]

assert train.shape == (712, 12), train.shape
assert test.shape == (179, 11), test.shape
assert train.columns.tolist() == expected_train_columns
assert test.columns.tolist() == expected_test_columns
assert set(train["Survived"].unique()) == {0, 1}
assert train["PassengerId"].is_unique and test["PassengerId"].is_unique
assert set(train["PassengerId"]).isdisjoint(set(test["PassengerId"]))
assert train["Survived"].value_counts().to_dict() == {0: 439, 1: 273}

X = train.drop(columns="Survived").reset_index(drop=True)
y = train["Survived"].astype(int).reset_index(drop=True)
X_test = test.copy().reset_index(drop=True)

data_summary = pd.DataFrame(
    {
        "rows": [len(train), len(test)],
        "columns": [train.shape[1], test.shape[1]],
        "Age_missing": [train["Age"].isna().sum(), test["Age"].isna().sum()],
        "Cabin_missing": [train["Cabin"].isna().sum(), test["Cabin"].isna().sum()],
    },
    index=["train", "test"],
)
display(data_summary)
display(train["Survived"].value_counts().sort_index().rename("count").to_frame())


In [ ]:
# 最小ベースライン：性別だけで予測（train全体の記述値。CV比較値ではない）
sex_baseline = (X["Sex"] == "female").astype(int)
baseline_metrics = {
    "F1": f1_score(y, sex_baseline),
    "Precision": precision_score(y, sex_baseline),
    "Recall": recall_score(y, sex_baseline),
    "Predicted positive": int(sex_baseline.sum()),
}
pd.Series(baseline_metrics, name="sex baseline (descriptive, not CV)")


## 2. リークを防ぐ特徴量変換

`TitanicFeatureEngineer` は、各foldの学習部分でのみ次を学習します。

- 年齢・運賃・乗船港の補完値
- Family、姓、Ticketの出現回数
- Family/Ticketの生存率target encoding

学習行のtarget encodingでは、自分自身の件数・ラベルと全体priorへの寄与も除外します。
検証・test行には学習foldで得た統計だけを写像し、未知グループは学習priorへ戻します。


In [ ]:
COMMON_TITLES = {"Mr", "Miss", "Mrs", "Master", "Dr", "Rev", "Officer", "Royalty"}


def normalize_title(name: pd.Series) -> pd.Series:
    title = name.fillna("").str.extract(r",\s*([^.]*)\.", expand=False).str.strip()
    title = title.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    title = title.replace({"Col": "Officer", "Major": "Officer", "Capt": "Officer"})
    title = title.replace(
        {"Lady": "Royalty", "Countess": "Royalty", "Sir": "Royalty", "Jonkheer": "Royalty", "Don": "Royalty", "Dona": "Royalty"}
    )
    return title.where(title.isin(COMMON_TITLES), "Rare")


def normalize_ticket(ticket: pd.Series) -> pd.Series:
    return ticket.fillna("UNKNOWN").astype(str).str.upper().str.replace(r"\s+", "", regex=True)


def ticket_prefix(ticket: pd.Series) -> pd.Series:
    prefix = (
        ticket.fillna("")
        .astype(str)
        .str.upper()
        .str.replace(r"\d", "", regex=True)
        .str.replace(r"[\s./]", "", regex=True)
    )
    return prefix.replace("", "NUMERIC")


class TitanicFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, profile: str = "role_te", target_smoothing: float = 10.0):
        self.profile = profile
        self.target_smoothing = target_smoothing

    def _base(self, X_frame: pd.DataFrame) -> pd.DataFrame:
        frame = X_frame.reset_index(drop=True).copy()
        out = pd.DataFrame(index=frame.index)
        out["Pclass"] = pd.to_numeric(frame["Pclass"], errors="coerce")
        out["Sex"] = frame["Sex"].fillna("Unknown").astype(str)
        out["Age"] = pd.to_numeric(frame["Age"], errors="coerce")
        out["SibSp"] = pd.to_numeric(frame["SibSp"], errors="coerce")
        out["Parch"] = pd.to_numeric(frame["Parch"], errors="coerce")
        out["Fare"] = pd.to_numeric(frame["Fare"], errors="coerce")
        out["Embarked"] = frame["Embarked"].fillna("Missing").astype(str)

        out["Title"] = normalize_title(frame["Name"])
        out["NameLength"] = frame["Name"].fillna("").str.len().astype(float)
        out["Surname"] = (
            frame["Name"].fillna("").str.split(",").str[0].str.upper().str.replace(r"[^A-Z]", "", regex=True)
        )
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["FamilySizeBand"] = pd.cut(
            out["FamilySize"], bins=[0, 1, 4, 7, np.inf], labels=["alone", "small", "medium", "large"]
        ).astype(str)
        out["AgeMissing"] = out["Age"].isna().astype(int)
        out["IsChildRaw"] = (out["Age"].fillna(99) < 16).astype(int)

        out["TicketKey"] = normalize_ticket(frame["Ticket"])
        out["TicketPrefix"] = ticket_prefix(frame["Ticket"])
        out["TicketDigits"] = frame["Ticket"].fillna("").str.count(r"\d").astype(float)
        out["CabinKnown"] = frame["Cabin"].notna().astype(int)
        out["CabinCount"] = frame["Cabin"].fillna("").str.split().str.len().where(frame["Cabin"].notna(), 0).astype(float)
        out["Deck"] = frame["Cabin"].fillna("U").astype(str).str[0].replace("", "U")
        out["FamilyKey"] = out["Surname"] + "_" + out["FamilySize"].fillna(-1).astype(int).astype(str)
        out["Role"] = np.where((out["Sex"] == "female") | (out["Age"].fillna(99) < 16), "female_child", "adult_male")
        return out

    def fit(self, X_frame: pd.DataFrame, y_target: pd.Series):
        if self.target_smoothing <= 0:
            raise ValueError("target_smoothing must be positive")
        base = self._base(X_frame)
        target = pd.Series(np.asarray(y_target, dtype=float), index=base.index)
        self.n_fit_ = len(base)
        self.target_total_ = float(target.sum())
        self.target_prior_ = float(target.mean())

        age_table = base.assign(_target_age=base["Age"])
        self.age_title_class_median_ = age_table.groupby(["Title", "Pclass", "Sex"])["_target_age"].median().to_dict()
        self.age_class_sex_median_ = age_table.groupby(["Pclass", "Sex"])["_target_age"].median().to_dict()
        self.age_global_median_ = float(base["Age"].median())
        self.fare_class_median_ = base.groupby("Pclass")["Fare"].median().to_dict()
        self.fare_global_median_ = float(base["Fare"].median())
        embarked_nonmissing = base.loc[base["Embarked"] != "Missing", "Embarked"]
        self.embarked_mode_ = str(embarked_nonmissing.mode().iloc[0])

        self.count_maps_ = {
            column: base[column].value_counts(dropna=False).to_dict()
            for column in ("FamilyKey", "TicketKey", "Surname")
        }
        self.target_states_ = {}
        for key_column in ("FamilyKey", "TicketKey"):
            group = pd.DataFrame({"key": base[key_column], "target": target}).groupby("key")["target"].agg(["sum", "count"])
            self.target_states_[(key_column, "all")] = (group["sum"].to_dict(), group["count"].to_dict())
            for role in ("female_child", "adult_male"):
                mask = base["Role"] == role
                role_group = pd.DataFrame({"key": base.loc[mask, key_column], "target": target.loc[mask]}).groupby("key")["target"].agg(["sum", "count"])
                self.target_states_[(key_column, role)] = (role_group["sum"].to_dict(), role_group["count"].to_dict())
        return self

    def _fill_age(self, base: pd.DataFrame) -> pd.Series:
        filled = base["Age"].copy()
        missing_indices = filled[filled.isna()].index
        for idx in missing_indices:
            key3 = (base.at[idx, "Title"], base.at[idx, "Pclass"], base.at[idx, "Sex"])
            key2 = (base.at[idx, "Pclass"], base.at[idx, "Sex"])
            value = self.age_title_class_median_.get(key3, np.nan)
            if pd.isna(value):
                value = self.age_class_sex_median_.get(key2, self.age_global_median_)
            filled.at[idx] = value
        return filled.fillna(self.age_global_median_)

    def _target_encode(
        self,
        base: pd.DataFrame,
        key_column: str,
        role: str,
        y_for_loo: pd.Series | None,
    ) -> tuple[pd.Series, pd.Series]:
        sum_map, count_map = self.target_states_[(key_column, role)]
        sums = base[key_column].map(sum_map).fillna(0.0).astype(float)
        counts = base[key_column].map(count_map).fillna(0.0).astype(float)

        if y_for_loo is not None:
            y_values = pd.Series(np.asarray(y_for_loo, dtype=float), index=base.index)
            exclude = pd.Series(True, index=base.index) if role == "all" else (base["Role"] == role)
            sums = sums - y_values * exclude.astype(float)
            counts = counts - exclude.astype(float)
            prior = (self.target_total_ - y_values) / max(self.n_fit_ - 1, 1)
        else:
            prior = pd.Series(self.target_prior_, index=base.index)

        encoded = (sums + self.target_smoothing * prior) / (counts + self.target_smoothing)
        return encoded.astype(float), (counts > 0).astype(int)

    def _transform(self, X_frame: pd.DataFrame, y_for_loo: pd.Series | None) -> pd.DataFrame:
        base = self._base(X_frame)
        result = pd.DataFrame(index=base.index)
        result["Pclass"] = base["Pclass"]
        result["Sex"] = base["Sex"]
        result["Age"] = self._fill_age(base)
        result["AgeMissing"] = base["AgeMissing"]
        result["SibSp"] = base["SibSp"]
        result["Parch"] = base["Parch"]
        result["FamilySize"] = base["FamilySize"]
        result["IsAlone"] = base["IsAlone"]
        result["IsChild"] = (result["Age"] < 16).astype(int)
        result["IsMother"] = ((base["Sex"] == "female") & (base["Parch"] > 0) & (result["Age"] >= 18) & (base["Title"] != "Miss")).astype(int)

        fare = base["Fare"].copy()
        for idx in fare[fare.isna()].index:
            fare.at[idx] = self.fare_class_median_.get(base.at[idx, "Pclass"], self.fare_global_median_)
        fare = fare.fillna(self.fare_global_median_)
        result["Fare"] = fare
        result["FareLog"] = np.log1p(fare.clip(lower=0))
        result["FarePerPerson"] = fare / base["FamilySize"].clip(lower=1)

        result["Embarked"] = base["Embarked"].replace("Missing", self.embarked_mode_)
        result["Title"] = base["Title"]
        result["NameLength"] = base["NameLength"]
        result["FamilySizeBand"] = base["FamilySizeBand"]
        result["TicketPrefix"] = base["TicketPrefix"]
        result["TicketDigits"] = base["TicketDigits"]
        result["CabinKnown"] = base["CabinKnown"]
        result["CabinCount"] = base["CabinCount"]
        result["Deck"] = base["Deck"]
        result["SexPclass"] = base["Sex"] + "_" + base["Pclass"].fillna(-1).astype(int).astype(str)
        result["TitlePclass"] = base["Title"] + "_" + base["Pclass"].fillna(-1).astype(int).astype(str)

        training_rows = y_for_loo is not None
        for key_column, output_name in (
            ("FamilyKey", "FamilyPeerCount"),
            ("TicketKey", "TicketPeerCount"),
            ("Surname", "SurnamePeerCount"),
        ):
            counts = base[key_column].map(self.count_maps_[key_column]).fillna(0).astype(float)
            if training_rows:
                counts = (counts - 1).clip(lower=0)
            result[output_name] = counts

        if self.profile in {"group_te", "role_te"}:
            for key_column, short in (("FamilyKey", "Family"), ("TicketKey", "Ticket")):
                encoded, covered = self._target_encode(base, key_column, "all", y_for_loo)
                result[f"{short}SurvivalTE"] = encoded
                result[f"{short}TECovered"] = covered

        if self.profile == "role_te":
            for key_column, short in (("FamilyKey", "Family"), ("TicketKey", "Ticket")):
                for role, role_short in (("female_child", "FemaleChild"), ("adult_male", "AdultMale")):
                    encoded, covered = self._target_encode(base, key_column, role, y_for_loo)
                    result[f"{short}{role_short}TE"] = encoded
                    result[f"{short}{role_short}Covered"] = covered

        assert "PassengerId" not in result.columns
        assert not any(column in result.columns for column in ("Name", "Ticket", "Cabin", "FamilyKey", "TicketKey", "Surname"))
        return result

    def fit_transform(self, X_frame: pd.DataFrame, y_target: pd.Series, **fit_params) -> pd.DataFrame:
        return self.fit(X_frame, y_target)._transform(X_frame, pd.Series(np.asarray(y_target), index=range(len(X_frame))))

    def transform(self, X_frame: pd.DataFrame) -> pd.DataFrame:
        return self._transform(X_frame, None)


In [ ]:
def build_model_matrices(
    X_fit: pd.DataFrame,
    y_fit: pd.Series,
    X_valid: pd.DataFrame,
    profile: str,
    target_smoothing: float,
):
    feature_engineer = TitanicFeatureEngineer(profile=profile, target_smoothing=target_smoothing)
    fit_features = feature_engineer.fit_transform(X_fit.reset_index(drop=True), y_fit.reset_index(drop=True))
    valid_features = feature_engineer.transform(X_valid.reset_index(drop=True))

    numeric_columns = fit_features.select_dtypes(include=np.number).columns.tolist()
    categorical_columns = [column for column in fit_features.columns if column not in numeric_columns]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_columns,
            ),
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="infrequent_if_exist",
                                min_frequency=3,
                                sparse_output=False,
                                dtype=np.float32,
                            ),
                        ),
                    ]
                ),
                categorical_columns,
            ),
        ],
        verbose_feature_names_out=False,
    )
    fit_matrix = np.asarray(preprocessor.fit_transform(fit_features), dtype=np.float32)
    valid_matrix = np.asarray(preprocessor.transform(valid_features), dtype=np.float32)
    assert np.isfinite(fit_matrix).all() and np.isfinite(valid_matrix).all()
    return fit_matrix, valid_matrix, feature_engineer, preprocessor


# 合成データで、自己ラベルが自分自身のLOO特徴へ直接入らないことを確認
synthetic = pd.DataFrame(
    {
        "PassengerId": [1, 2, 3],
        "Pclass": [3, 3, 1],
        "Name": ["Smith, Mr. A", "Smith, Mrs. B", "Other, Mr. C"],
        "Sex": ["male", "female", "male"],
        "Age": [30.0, 28.0, 40.0],
        "SibSp": [1, 1, 0],
        "Parch": [0, 0, 0],
        "Ticket": ["A/1", "A/1", "B/2"],
        "Fare": [20.0, 20.0, 80.0],
        "Cabin": [np.nan, np.nan, "C10"],
        "Embarked": ["S", "S", "C"],
    }
)
y_a = pd.Series([0, 1, 0])
y_b = pd.Series([1, 1, 0])
feature_a = TitanicFeatureEngineer(profile="role_te", target_smoothing=5).fit_transform(synthetic, y_a)
feature_b = TitanicFeatureEngineer(profile="role_te", target_smoothing=5).fit_transform(synthetic, y_b)
assert np.isclose(feature_a.loc[0, "TicketSurvivalTE"], feature_b.loc[0, "TicketSurvivalTE"])
assert feature_a.loc[2, "TicketPeerCount"] == 0

unseen = synthetic.iloc[[2]].copy()
unseen["Ticket"] = "UNSEEN"
fitted = TitanicFeatureEngineer(profile="role_te", target_smoothing=5).fit(synthetic.iloc[:2], y_a.iloc[:2])
unseen_features = fitted.transform(unseen)
assert unseen_features.loc[0, "TicketTECovered"] == 0
assert clone(TitanicFeatureEngineer(profile="group_te", target_smoothing=7)).get_params() == {
    "profile": "group_te",
    "target_smoothing": 7,
}
print("特徴量変換のリーク防止テスト: OK")


## 3. F1閾値とモデル探索の共通処理

各trialでは、inner OOF確率を作った後、あるfoldの閾値を**それ以外のfold**で決めてから保持foldを採点します。
これにより、同じ予測を使って閾値を選択・評価する楽観性を減らします。


In [ ]:
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01 if FAST_MODE else 0.002)
TUNED_MODELS = ("catboost", "lightgbm", "xgboost", "extra_trees", "random_forest")
BOOST_MODELS = {"catboost", "lightgbm", "xgboost"}
PROFILE_CHOICES = ("base", "group_te", "role_te")


def find_plateau_threshold(y_true: pd.Series | np.ndarray, probabilities: np.ndarray, tolerance: float = 0.002):
    y_array = np.asarray(y_true, dtype=int)
    scores = np.array([f1_score(y_array, probabilities >= threshold) for threshold in THRESHOLD_GRID])
    best_score = float(scores.max())
    acceptable = THRESHOLD_GRID[scores >= best_score - tolerance]
    threshold = float(np.median(acceptable))
    return threshold, float(f1_score(y_array, probabilities >= threshold))


def cross_fitted_threshold_summary(y_true, probabilities, fold_ids):
    y_array = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    fold_ids = np.asarray(fold_ids, dtype=int)
    fold_scores, thresholds = [], []
    for fold_id in np.unique(fold_ids):
        validation_mask = fold_ids == fold_id
        threshold, _ = find_plateau_threshold(y_array[~validation_mask], probabilities[~validation_mask])
        score = f1_score(y_array[validation_mask], probabilities[validation_mask] >= threshold)
        thresholds.append(threshold)
        fold_scores.append(score)
    mean_score = float(np.mean(fold_scores))
    std_score = float(np.std(fold_scores, ddof=1)) if len(fold_scores) > 1 else 0.0
    return {
        "mean_f1": mean_score,
        "std_f1": std_score,
        "robust_score": mean_score - 0.25 * std_score,
        "threshold": float(np.median(thresholds)),
        "fold_scores": fold_scores,
        "fold_thresholds": thresholds,
    }


def trial_parameters(trial: optuna.Trial, model_name: str) -> dict:
    params = {
        "feature_profile": trial.suggest_categorical("feature_profile", PROFILE_CHOICES),
        "target_smoothing": trial.suggest_float("target_smoothing", 3.0, 30.0, log=True),
    }
    if model_name == "catboost":
        params.update(
            iterations=trial.suggest_int("iterations", 300, 1200, step=100),
            depth=trial.suggest_int("depth", 4, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.015, 0.15, log=True),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
            random_strength=trial.suggest_float("random_strength", 0.05, 3.0, log=True),
            bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 2.0),
            scale_pos_weight=trial.suggest_float("scale_pos_weight", 0.8, 1.8),
        )
    elif model_name == "lightgbm":
        params.update(
            n_estimators=trial.suggest_int("n_estimators", 250, 1200, step=50),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
            num_leaves=trial.suggest_int("num_leaves", 7, 31),
            max_depth=trial.suggest_int("max_depth", 3, 8),
            min_child_samples=trial.suggest_int("min_child_samples", 10, 50),
            subsample=trial.suggest_float("subsample", 0.65, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.65, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 3.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 5.0, log=True),
            scale_pos_weight=trial.suggest_float("scale_pos_weight", 0.8, 1.8),
        )
    elif model_name == "xgboost":
        params.update(
            n_estimators=trial.suggest_int("n_estimators", 250, 1200, step=50),
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
            max_depth=trial.suggest_int("max_depth", 2, 6),
            min_child_weight=trial.suggest_float("min_child_weight", 1.0, 12.0),
            gamma=trial.suggest_float("gamma", 1e-4, 3.0, log=True),
            subsample=trial.suggest_float("subsample", 0.65, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.65, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 3.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
            scale_pos_weight=trial.suggest_float("scale_pos_weight", 0.8, 1.8),
        )
    elif model_name in {"extra_trees", "random_forest"}:
        params.update(
            n_estimators=trial.suggest_int("n_estimators", 400, 1200, step=100),
            max_depth=trial.suggest_categorical("max_depth", [4, 5, 6, 8, 10, 12, None]),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 12),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 8),
            max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", 0.5, 0.8]),
            class_weight=trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
        )
    else:
        raise ValueError(model_name)
    return params


def build_model(model_name: str, params: dict, seed: int):
    model_params = {key: value for key, value in params.items() if key not in {"feature_profile", "target_smoothing"}}
    if model_name == "catboost":
        return CatBoostClassifier(
            **model_params,
            loss_function="Logloss",
            random_seed=seed,
            verbose=False,
            allow_writing_files=False,
            thread_count=MODEL_THREADS,
        )
    if model_name == "lightgbm":
        return LGBMClassifier(
            **model_params,
            objective="binary",
            random_state=seed,
            n_jobs=MODEL_THREADS,
            verbosity=-1,
            subsample_freq=1,
            deterministic=True,
            force_col_wise=True,
        )
    if model_name == "xgboost":
        return XGBClassifier(
            **model_params,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=seed,
            n_jobs=MODEL_THREADS,
            tree_method="hist",
        )
    if model_name == "extra_trees":
        return ExtraTreesClassifier(**model_params, random_state=seed, n_jobs=MODEL_THREADS)
    if model_name == "random_forest":
        return RandomForestClassifier(**model_params, random_state=seed, n_jobs=MODEL_THREADS)
    if model_name == "logistic":
        return LogisticRegression(C=0.5, max_iter=3000, solver="liblinear", random_state=seed)
    raise ValueError(model_name)


LOGISTIC_PARAMS = {"feature_profile": "role_te", "target_smoothing": 10.0}


def positive_probability(model, matrix):
    classes = np.asarray(model.classes_)
    positive_index = np.flatnonzero(classes == 1)
    if len(positive_index) != 1:
        raise ValueError(f"正例1の確率列を特定できません: classes={classes}")
    return model.predict_proba(matrix)[:, int(positive_index[0])]


In [ ]:
def make_cv_splits(X_frame, y_target, n_splits: int, seed: int):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(splitter.split(X_frame, y_target))


def fit_predict_fold(model_name, params, X_fit, y_fit, X_valid, seed):
    fit_matrix, valid_matrix, _, _ = build_model_matrices(
        X_fit,
        y_fit,
        X_valid,
        profile=params["feature_profile"],
        target_smoothing=float(params["target_smoothing"]),
    )
    model = build_model(model_name, params, seed)
    model.fit(fit_matrix, y_fit)
    return positive_probability(model, valid_matrix)


def make_oof_predictions(model_name, params, X_frame, y_target, splits, seed):
    oof = np.full(len(X_frame), np.nan, dtype=float)
    fold_ids = np.full(len(X_frame), -1, dtype=int)
    for fold_id, (fit_index, valid_index) in enumerate(splits):
        probabilities = fit_predict_fold(
            model_name,
            params,
            X_frame.iloc[fit_index].reset_index(drop=True),
            y_target.iloc[fit_index].reset_index(drop=True),
            X_frame.iloc[valid_index].reset_index(drop=True),
            seed + fold_id,
        )
        oof[valid_index] = probabilities
        fold_ids[valid_index] = fold_id
    assert np.isfinite(oof).all() and (fold_ids >= 0).all()
    return oof, fold_ids


def make_oof_with_external_predictions(
    model_name,
    params,
    X_frame,
    y_target,
    splits,
    X_external,
    seed,
):
    oof = np.full(len(X_frame), np.nan, dtype=float)
    fold_ids = np.full(len(X_frame), -1, dtype=int)
    external_probabilities = []
    for fold_id, (fit_index, valid_index) in enumerate(splits):
        X_fit = X_frame.iloc[fit_index].reset_index(drop=True)
        y_fit = y_target.iloc[fit_index].reset_index(drop=True)
        X_valid = X_frame.iloc[valid_index].reset_index(drop=True)
        feature_engineer = TitanicFeatureEngineer(
            profile=params["feature_profile"],
            target_smoothing=float(params["target_smoothing"]),
        )
        fit_features = feature_engineer.fit_transform(X_fit, y_fit)
        valid_features = feature_engineer.transform(X_valid)
        external_features = feature_engineer.transform(X_external.reset_index(drop=True))

        numeric_columns = fit_features.select_dtypes(include=np.number).columns.tolist()
        categorical_columns = [column for column in fit_features.columns if column not in numeric_columns]
        preprocessor = ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                    numeric_columns,
                ),
                (
                    "categorical",
                    Pipeline(
                        [
                            ("imputer", SimpleImputer(strategy="most_frequent")),
                            (
                                "onehot",
                                OneHotEncoder(
                                    handle_unknown="infrequent_if_exist",
                                    min_frequency=3,
                                    sparse_output=False,
                                    dtype=np.float32,
                                ),
                            ),
                        ]
                    ),
                    categorical_columns,
                ),
            ],
            verbose_feature_names_out=False,
        )
        fit_matrix = np.asarray(preprocessor.fit_transform(fit_features), dtype=np.float32)
        valid_matrix = np.asarray(preprocessor.transform(valid_features), dtype=np.float32)
        external_matrix = np.asarray(preprocessor.transform(external_features), dtype=np.float32)
        model = build_model(model_name, params, seed + fold_id)
        model.fit(fit_matrix, y_fit)
        oof[valid_index] = positive_probability(model, valid_matrix)
        external_probabilities.append(positive_probability(model, external_matrix))
        fold_ids[valid_index] = fold_id
    assert np.isfinite(oof).all() and (fold_ids >= 0).all()
    return oof, fold_ids, np.mean(external_probabilities, axis=0)


def study_trial_count(model_name: str, phase: str) -> int:
    if phase == "nested":
        return CONFIG.nested_trials_boost if model_name in BOOST_MODELS else CONFIG.nested_trials_forest
    return CONFIG.final_trials_boost if model_name in BOOST_MODELS else CONFIG.final_trials_forest


def run_single_cv_study(model_name, X_frame, y_target, splits, seed, n_trials):
    def objective(trial):
        params = trial_parameters(trial, model_name)
        oof, fold_ids = make_oof_predictions(model_name, params, X_frame, y_target, splits, seed)
        return cross_fitted_threshold_summary(y_target, oof, fold_ids)["robust_score"]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=n_trials, n_jobs=OPTUNA_JOBS, show_progress_bar=False)
    return study


def optimize_blend(y_target, probability_frame, fold_ids, n_trials, seed):
    probability_matrix = probability_frame.to_numpy(dtype=float)
    columns = probability_frame.columns.tolist()

    def objective(trial):
        raw_weights = np.array([trial.suggest_float(f"w_{column}", 0.0, 1.0) for column in columns])
        if raw_weights.sum() <= 1e-12:
            return 0.0
        probabilities = probability_matrix @ (raw_weights / raw_weights.sum())
        return cross_fitted_threshold_summary(y_target, probabilities, fold_ids)["robust_score"]

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    weights = np.array([study.best_params[f"w_{column}"] for column in columns], dtype=float)
    weights /= weights.sum()
    blended = probability_matrix @ weights
    summary = cross_fitted_threshold_summary(y_target, blended, fold_ids)
    return dict(zip(columns, weights)), blended, summary


def cross_fit_stacking(y_target, probability_frame, fold_ids):
    matrix = probability_frame.to_numpy(dtype=float)
    y_array = np.asarray(y_target, dtype=int)
    candidates = [0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
    best = None
    for c_value in candidates:
        meta_oof = np.full(len(y_array), np.nan)
        for fold_id in np.unique(fold_ids):
            valid_mask = np.asarray(fold_ids) == fold_id
            meta_model = LogisticRegression(C=c_value, max_iter=3000, solver="liblinear", random_state=SEED)
            meta_model.fit(matrix[~valid_mask], y_array[~valid_mask])
            meta_oof[valid_mask] = positive_probability(meta_model, matrix[valid_mask])
        summary = cross_fitted_threshold_summary(y_array, meta_oof, fold_ids)
        if best is None or summary["robust_score"] > best["summary"]["robust_score"]:
            best = {"C": c_value, "meta_oof": meta_oof, "summary": summary}
    return best


## 4. Nested CVによる方式比較

outer検証行は、特徴量統計・Optuna・閾値・blend重み・stackingメタモデルの選択に一切使いません。
表示されるnested F1を、性能比較の主な根拠とします。


In [ ]:
def run_nested_evaluation(X_frame, y_target):
    fold_results = []
    selection_details = []
    all_models = list(TUNED_MODELS) + ["logistic"]
    started = time.perf_counter()
    outer_counter = 0

    for repeat in range(CONFIG.outer_repeats):
        outer_seed = SEED + repeat * 1009
        outer_splits = make_cv_splits(X_frame, y_target, CONFIG.outer_splits, outer_seed)
        for outer_fold, (outer_fit_idx, outer_valid_idx) in enumerate(outer_splits):
            outer_counter += 1
            X_outer_fit = X_frame.iloc[outer_fit_idx].reset_index(drop=True)
            y_outer_fit = y_target.iloc[outer_fit_idx].reset_index(drop=True)
            X_outer_valid = X_frame.iloc[outer_valid_idx].reset_index(drop=True)
            y_outer_valid = y_target.iloc[outer_valid_idx].reset_index(drop=True)
            inner_seed = outer_seed + outer_fold + 17
            inner_splits = make_cv_splits(X_outer_fit, y_outer_fit, CONFIG.inner_splits, inner_seed)

            inner_probabilities = {}
            outer_probabilities = {}
            outer_stack_probabilities = {}
            inner_summaries = {}
            selected_params = {}
            common_fold_ids = None

            for model_name in TUNED_MODELS:
                study = run_single_cv_study(
                    model_name,
                    X_outer_fit,
                    y_outer_fit,
                    inner_splits,
                    seed=inner_seed + 101 * (TUNED_MODELS.index(model_name) + 1),
                    n_trials=study_trial_count(model_name, "nested"),
                )
                params = dict(study.best_trial.params)
                inner_oof, fold_ids, stack_outer_probability = make_oof_with_external_predictions(
                    model_name,
                    params,
                    X_outer_fit,
                    y_outer_fit,
                    inner_splits,
                    X_outer_valid,
                    inner_seed + 2000,
                )
                inner_probabilities[model_name] = inner_oof
                inner_summaries[model_name] = cross_fitted_threshold_summary(y_outer_fit, inner_oof, fold_ids)
                selected_params[model_name] = params
                outer_probabilities[model_name] = fit_predict_fold(
                    model_name, params, X_outer_fit, y_outer_fit, X_outer_valid, inner_seed + 3000
                )
                outer_stack_probabilities[model_name] = stack_outer_probability
                common_fold_ids = fold_ids

            logistic_oof, logistic_fold_ids, logistic_stack_outer = make_oof_with_external_predictions(
                "logistic",
                LOGISTIC_PARAMS,
                X_outer_fit,
                y_outer_fit,
                inner_splits,
                X_outer_valid,
                inner_seed + 4000,
            )
            inner_probabilities["logistic"] = logistic_oof
            inner_summaries["logistic"] = cross_fitted_threshold_summary(y_outer_fit, logistic_oof, logistic_fold_ids)
            selected_params["logistic"] = dict(LOGISTIC_PARAMS)
            outer_probabilities["logistic"] = fit_predict_fold(
                "logistic", LOGISTIC_PARAMS, X_outer_fit, y_outer_fit, X_outer_valid, inner_seed + 5000
            )
            outer_stack_probabilities["logistic"] = logistic_stack_outer

            inner_frame = pd.DataFrame(inner_probabilities)
            outer_frame = pd.DataFrame(outer_probabilities)
            outer_stack_frame = pd.DataFrame(outer_stack_probabilities)
            best_single = max(all_models, key=lambda name: inner_summaries[name]["robust_score"])
            single_threshold = inner_summaries[best_single]["threshold"]
            single_labels = (outer_frame[best_single].to_numpy() >= single_threshold).astype(int)

            blend_weights, _, blend_summary = optimize_blend(
                y_outer_fit,
                inner_frame,
                common_fold_ids,
                n_trials=max(15, CONFIG.blend_trials // 10),
                seed=inner_seed + 6000,
            )
            blend_outer = sum(outer_frame[name].to_numpy() * weight for name, weight in blend_weights.items())
            blend_labels = (blend_outer >= blend_summary["threshold"]).astype(int)

            stack_info = cross_fit_stacking(y_outer_fit, inner_frame, common_fold_ids)
            final_meta = LogisticRegression(
                C=stack_info["C"], max_iter=3000, solver="liblinear", random_state=inner_seed
            )
            final_meta.fit(inner_frame.to_numpy(), y_outer_fit)
            stack_outer = positive_probability(final_meta, outer_stack_frame[inner_frame.columns].to_numpy())
            stack_labels = (stack_outer >= stack_info["summary"]["threshold"]).astype(int)

            for method, labels in (
                ("best_single", single_labels),
                ("weighted_blend", blend_labels),
                ("crossfit_stacking", stack_labels),
            ):
                fold_results.append(
                    {
                        "repeat": repeat,
                        "outer_fold": outer_fold,
                        "method": method,
                        "f1": f1_score(y_outer_valid, labels),
                        "precision": precision_score(y_outer_valid, labels, zero_division=0),
                        "recall": recall_score(y_outer_valid, labels, zero_division=0),
                    }
                )
            selection_details.append(
                {
                    "repeat": repeat,
                    "outer_fold": outer_fold,
                    "best_single": best_single,
                    "single_threshold": single_threshold,
                    "blend_threshold": blend_summary["threshold"],
                    "stack_C": stack_info["C"],
                    "stack_threshold": stack_info["summary"]["threshold"],
                }
            )
            elapsed_minutes = (time.perf_counter() - started) / 60
            print(
                f"outer {outer_counter}/{CONFIG.outer_splits * CONFIG.outer_repeats} 完了 "
                f"({elapsed_minutes:.1f} min, best={best_single})"
            )

    return pd.DataFrame(fold_results), pd.DataFrame(selection_details)


nested_fold_results, nested_selection_details = run_nested_evaluation(X, y)
nested_summary = (
    nested_fold_results.groupby("method")
    .agg(mean_f1=("f1", "mean"), std_f1=("f1", "std"), mean_precision=("precision", "mean"), mean_recall=("recall", "mean"))
    .assign(robust_score=lambda frame: frame["mean_f1"] - 0.25 * frame["std_f1"].fillna(0))
    .sort_values("robust_score", ascending=False)
)
display(nested_summary)
display(nested_selection_details)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=nested_fold_results, x="method", y="f1", ax=axes[0])
axes[0].set_title("Nested outer-fold F1")
axes[0].tick_params(axis="x", rotation=15)

selection_counts = nested_selection_details["best_single"].value_counts()
selection_counts.plot(kind="bar", ax=axes[1], color="steelblue")
axes[1].set_title("Best single model selected by inner CV")
axes[1].set_ylabel("outer fold count")
plt.tight_layout()
plt.show()


## 5. 全train上の最終探索

ここからのOOF値は、全trainを使って最終パラメータを選ぶための値です。モデル選択を含むため、
nested outer F1より楽観的になり得ます。最終提出の設定には使いますが、公平な性能推定とは区別します。


In [ ]:
def evaluate_params_repeated(model_name, params, X_frame, y_target, seeds):
    repeat_summaries = []
    repeat_oof = []
    for cv_seed in seeds:
        splits = make_cv_splits(X_frame, y_target, 3 if FAST_MODE else 5, cv_seed)
        oof, fold_ids = make_oof_predictions(model_name, params, X_frame, y_target, splits, cv_seed + 7000)
        repeat_oof.append(oof)
        repeat_summaries.append(cross_fitted_threshold_summary(y_target, oof, fold_ids))
    robust_scores = [summary["robust_score"] for summary in repeat_summaries]
    return {
        "objective": float(np.mean(robust_scores) - 0.25 * np.std(robust_scores)),
        "repeat_summaries": repeat_summaries,
        "mean_oof": np.mean(repeat_oof, axis=0),
    }


def run_final_study(model_name, X_frame, y_target, seed):
    def objective(trial):
        params = trial_parameters(trial, model_name)
        return evaluate_params_repeated(
            model_name, params, X_frame, y_target, CONFIG.final_cv_seeds
        )["objective"]

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(
        objective,
        n_trials=study_trial_count(model_name, "final"),
        n_jobs=OPTUNA_JOBS,
        show_progress_bar=False,
    )
    return study


final_params = {}
final_oof_probabilities = {}
final_model_rows = []
final_repeat_details = {}
final_started = time.perf_counter()

for model_index, model_name in enumerate(TUNED_MODELS):
    study = run_final_study(model_name, X, y, seed=SEED + 10000 + model_index * 101)
    params = dict(study.best_trial.params)
    evaluation = evaluate_params_repeated(model_name, params, X, y, CONFIG.final_cv_seeds)
    mean_oof = evaluation["mean_oof"]
    meta_splits = make_cv_splits(X, y, 3 if FAST_MODE else 5, SEED + 16000)
    meta_fold_ids = np.full(len(X), -1, dtype=int)
    for fold_id, (_, valid_index) in enumerate(meta_splits):
        meta_fold_ids[valid_index] = fold_id
    summary = cross_fitted_threshold_summary(y, mean_oof, meta_fold_ids)
    final_params[model_name] = params
    final_oof_probabilities[model_name] = mean_oof
    final_repeat_details[model_name] = evaluation["repeat_summaries"]
    final_model_rows.append(
        {
            "model": model_name,
            "mean_f1": summary["mean_f1"],
            "std_f1": summary["std_f1"],
            "robust_score": summary["robust_score"],
            "threshold": summary["threshold"],
            "average_precision": average_precision_score(y, mean_oof),
        }
    )
    print(f"最終探索 {model_name} 完了 ({(time.perf_counter() - final_started) / 60:.1f} min)")

logistic_evaluation = evaluate_params_repeated("logistic", LOGISTIC_PARAMS, X, y, CONFIG.final_cv_seeds)
logistic_oof = logistic_evaluation["mean_oof"]
meta_splits = make_cv_splits(X, y, 3 if FAST_MODE else 5, SEED + 16000)
meta_fold_ids = np.full(len(X), -1, dtype=int)
for fold_id, (_, valid_index) in enumerate(meta_splits):
    meta_fold_ids[valid_index] = fold_id
logistic_summary = cross_fitted_threshold_summary(y, logistic_oof, meta_fold_ids)
final_params["logistic"] = dict(LOGISTIC_PARAMS)
final_oof_probabilities["logistic"] = logistic_oof
final_model_rows.append(
    {
        "model": "logistic",
        "mean_f1": logistic_summary["mean_f1"],
        "std_f1": logistic_summary["std_f1"],
        "robust_score": logistic_summary["robust_score"],
        "threshold": logistic_summary["threshold"],
        "average_precision": average_precision_score(y, logistic_oof),
    }
)

final_model_results = pd.DataFrame(final_model_rows).sort_values("robust_score", ascending=False).reset_index(drop=True)
display(final_model_results)
final_parameter_table = pd.DataFrame(
    {
        "selected_parameters": {
            model_name: repr(parameters)
            for model_name, parameters in final_params.items()
        }
    }
)
with pd.option_context("display.max_colwidth", 180):
    display(final_parameter_table)


In [ ]:
final_oof_frame = pd.DataFrame(final_oof_probabilities)

blend_weights, blend_oof, blend_summary = optimize_blend(
    y,
    final_oof_frame,
    meta_fold_ids,
    n_trials=CONFIG.blend_trials,
    seed=SEED + 17000,
)
stack_info = cross_fit_stacking(y, final_oof_frame, meta_fold_ids)

best_single_name = str(final_model_results.iloc[0]["model"])
best_single_threshold = float(final_model_results.iloc[0]["threshold"])

strategy_results = pd.DataFrame(
    [
        {
            "strategy": f"best_single:{best_single_name}",
            "mean_f1": float(final_model_results.iloc[0]["mean_f1"]),
            "std_f1": float(final_model_results.iloc[0]["std_f1"]),
            "robust_score": float(final_model_results.iloc[0]["robust_score"]),
            "threshold": best_single_threshold,
        },
        {
            "strategy": "weighted_blend",
            "mean_f1": blend_summary["mean_f1"],
            "std_f1": blend_summary["std_f1"],
            "robust_score": blend_summary["robust_score"],
            "threshold": blend_summary["threshold"],
        },
        {
            "strategy": "crossfit_stacking",
            "mean_f1": stack_info["summary"]["mean_f1"],
            "std_f1": stack_info["summary"]["std_f1"],
            "robust_score": stack_info["summary"]["robust_score"],
            "threshold": stack_info["summary"]["threshold"],
        },
    ]
).sort_values("robust_score", ascending=False)

display(strategy_results)
display(pd.Series(blend_weights, name="blend weight").sort_values(ascending=False).to_frame())
print(f"Stacking C={stack_info['C']}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=final_model_results, x="model", y="robust_score", ax=axes[0], color="steelblue")
axes[0].set_title("Final selection-inclusive robust F1")
axes[0].tick_params(axis="x", rotation=30)

correlation = final_oof_frame.corr()
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag", vmin=0.5, vmax=1.0, ax=axes[1])
axes[1].set_title("OOF probability correlation")
plt.tight_layout()
plt.show()


## 6. 全train再学習とtest推論

モデル、特徴profile、ハイパーパラメータ、閾値、blend重み、stacking設定はここまでで凍結済みです。
ここで初めてtest特徴量を変換します。各モデルは全trainを使って複数seedで学習し、test確率を平均します。


In [ ]:
def fit_full_seed_ensemble(model_name, params, X_frame, y_target, X_inference, seeds):
    fit_matrix, inference_matrix, _, _ = build_model_matrices(
        X_frame.reset_index(drop=True),
        y_target.reset_index(drop=True),
        X_inference.reset_index(drop=True),
        profile=params["feature_profile"],
        target_smoothing=float(params["target_smoothing"]),
    )
    effective_seeds = seeds[:1] if model_name == "logistic" else seeds
    probabilities = []
    for seed in effective_seeds:
        model = build_model(model_name, params, seed)
        model.fit(fit_matrix, y_target)
        probabilities.append(positive_probability(model, inference_matrix))
    return np.mean(probabilities, axis=0)


final_test_probabilities = {}
stack_test_probabilities = {}
for model_name in final_oof_frame.columns:
    final_test_probabilities[model_name] = fit_full_seed_ensemble(
        model_name,
        final_params[model_name],
        X,
        y,
        X_test,
        CONFIG.final_fit_seeds,
    )
    stack_seed_predictions = []
    for cv_seed in CONFIG.final_cv_seeds:
        stack_splits = make_cv_splits(X, y, 3 if FAST_MODE else 5, cv_seed)
        _, _, stack_test_probability = make_oof_with_external_predictions(
            model_name,
            final_params[model_name],
            X,
            y,
            stack_splits,
            X_test,
            cv_seed + 19000,
        )
        stack_seed_predictions.append(stack_test_probability)
    stack_test_probabilities[model_name] = np.mean(stack_seed_predictions, axis=0)
    print(f"全train再学習: {model_name} 完了")

final_test_frame = pd.DataFrame(final_test_probabilities)
stack_test_frame = pd.DataFrame(stack_test_probabilities)[final_oof_frame.columns]
single_test_probability = final_test_frame[best_single_name].to_numpy()
blend_test_probability = sum(final_test_frame[name].to_numpy() * weight for name, weight in blend_weights.items())

final_meta_model = LogisticRegression(
    C=stack_info["C"], max_iter=3000, solver="liblinear", random_state=SEED
)
final_meta_model.fit(final_oof_frame.to_numpy(), y)
stack_test_probability = positive_probability(final_meta_model, stack_test_frame.to_numpy())

submission_predictions = {
    "submission_best_single_nested.csv": (single_test_probability >= best_single_threshold).astype(int),
    "submission_weighted_blend_nested.csv": (blend_test_probability >= blend_summary["threshold"]).astype(int),
    "submission_crossfit_stacking.csv": (stack_test_probability >= stack_info["summary"]["threshold"]).astype(int),
}

pd.DataFrame(
    {
        name: {
            "predicted_survivors": int(predictions.sum()),
            "predicted_rate": float(predictions.mean()),
        }
        for name, predictions in submission_predictions.items()
    }
).T


## 7. 提出CSVの保存と再読込検査

既存ファイルを黙って上書きしません。同名ファイルが存在する場合は、内容が完全一致するときだけ再利用し、
異なる場合は `FileExistsError` で停止します。


In [ ]:
def validate_submission(frame: pd.DataFrame, test_frame: pd.DataFrame):
    assert frame.columns.tolist() == ["PassengerId", "Survived"]
    assert len(frame) == len(test_frame) == 179
    assert frame["PassengerId"].is_unique
    assert not frame.isna().any().any()
    assert frame["PassengerId"].tolist() == test_frame["PassengerId"].tolist()
    assert set(frame["PassengerId"]) == set(test_frame["PassengerId"])
    assert set(frame["Survived"].unique()).issubset({0, 1})
    assert pd.api.types.is_integer_dtype(frame["Survived"])


def safe_save_submission(path: Path, predictions: np.ndarray, test_frame: pd.DataFrame):
    frame = pd.DataFrame(
        {
            "PassengerId": test_frame["PassengerId"].astype(int),
            "Survived": np.asarray(predictions, dtype=int),
        }
    )
    validate_submission(frame, test_frame)
    if path.exists():
        existing = pd.read_csv(path)
        validate_submission(existing, test_frame)
        if not existing.equals(frame):
            raise FileExistsError(f"既存ファイルと内容が異なるため上書きしません: {path}")
        print(f"既存ファイルと一致: {path.name}")
    else:
        frame.to_csv(path, index=False, mode="x")
        print(f"新規保存: {path.name}")
    reloaded = pd.read_csv(path)
    validate_submission(reloaded, test_frame)
    assert reloaded.equals(frame)
    return frame


saved_submissions = {}
for filename, predictions in submission_predictions.items():
    saved_submissions[filename] = safe_save_submission(DATA_DIR / filename, predictions, X_test)

comparison = pd.DataFrame(
    {name: frame["Survived"].to_numpy() for name, frame in saved_submissions.items()}
)
pairwise_disagreement = pd.DataFrame(index=comparison.columns, columns=comparison.columns, dtype=int)
for left in comparison.columns:
    for right in comparison.columns:
        pairwise_disagreement.loc[left, right] = int((comparison[left] != comparison[right]).sum())
display(pairwise_disagreement.astype(int))
print("提出CSV 3件の形式・ID・ラベル検査: OK")


## 結論と提出時の説明

- 公平な性能推定には `nested_summary` を使用する。
- 最終設定の比較には `final_model_results` と `strategy_results` を使用する。
- 3提出は、最良単一モデル、OOF重み付きblend、base OOFとmeta cross-fitを組み合わせたstackingである。
- すべての予測は配布trainから学習したモデルの確率と、train-onlyで決めた閾値から生成した。
- test正解、外部データ、手作業による個別ラベル変更、LLM推論は使用していない。

提出前には、上の実行結果に表示された採用モデル、特徴profile、ハイパーパラメータ、閾値、blend重み、
予測生存者数を確認してください。
